## Simulation

In [ ]:
## Import modules
%reload_ext autoreload
%autoreload 2
from interpolation_uncertainty.core.loadfile import load_file
from interpolation_uncertainty.processors.rasterProcessor import RasterProcessor
from interpolation_uncertainty.utils import utils

import matplotlib.pyplot as plt
from pathlib import Path
from pptx import Presentation
from pptx.util import Inches
from PIL import Image
import io


In [ ]:
def process_file(filename: str):

    # Read filename and load bathy depth
    bathy_data = load_file(filename)
    max_multiple = 1
    current_multiple = 1
    windowing = 'hann'
    # methods = ['amp_v1', 'psd_v1', 'amp_v2', 'psd_n',
    #            'psd_lf', 'psd_df', 'spectrum',]
    methods = ['spatial_std', 'spatial_diff','spatial_gaussian']
    # methods = ['amp_v1', 'psd_v1', 'amp_v2', 'psd_n',
    #            'psd_lf', 'psd_df', 'spectrum','spatial_std',
    #            'spatial_diff','spatial_gaussian']
    linespace_list = [128]
    # linespace_list = [64]
    prs = Presentation()
    for method in methods:
        for l_space in linespace_list:
            linespacing = l_space
            bp_stats = []

            raster_processor = RasterProcessor(linespacing_meters=linespacing,
                                max_multiple=max_multiple,
                                multiple=current_multiple)

            # ## Get boxplot statistics for each case
            case_1 = utils.case_1_bp(bathy_data, raster_processor, method)
            bp_stats.append(case_1)

            case_2_toprow = utils.case_2_bp(bathy_data, raster_processor, method, base_row='top')
            bp_stats.append(case_2_toprow)

            case_2_midrow = utils.case_2_bp(bathy_data, raster_processor, method, base_row='middle')
            bp_stats.append(case_2_midrow)

            # Case 3: Use either left or right column/along track data as proxy for across track
            # Window size for each sample is equal to the linespacing
            case_3_left = utils.case_3_bp(bathy_data, raster_processor, method, selected_column='left')
            bp_stats.append(case_3_left)
            case_3_right = utils.case_3_bp(bathy_data, raster_processor, method, selected_column='right')
            bp_stats.append(case_3_right)

            # Case 4: Use both left or right column/along track data as proxy for across track
            # Bias value determine share of left/right column in creating the proxy surface
            # case_4_half = utils.case_4_bp(bathy_data, raster_processor, method, bias=0.5)
            # bp_stats.append(case_4_half)

            bp_stats = [item for row in bp_stats for item in row]
            # print(bp_stats)
            fig1, ax1 = plt.subplots(figsize=(10,6))
            ax1.bxp(bp_stats, patch_artist=True, showfliers=False)
            # ax1.bxp(bp_stats, patch_artist=True)
            xtick_labels_objects = ax1.get_xticklabels()
            xtick_labels_text = [label.get_text() for label in xtick_labels_objects]
            plt.xticks(ax1.get_xticks(), xtick_labels_text, rotation=45, ha='right')
            plt.ylabel("Uncertainty")
            plt.xlabel("Linespacing (meters)")
            plt.title(f"""Surface: {Path(filename).stem}, {linespacing}m linespacing \n using {method}""")
            plt.tight_layout()
            
            image_stream = io.BytesIO()
            plt.savefig(image_stream, format='png')
            plt.close(fig1)
            
            blank_slide_layout = prs.slide_layouts[6] 
            slide = prs.slides.add_slide(blank_slide_layout)
            left = Inches(0)
            top = Inches(0)
            width = Inches(10)
            height = Inches(7)

            pic = slide.shapes.add_picture(image_stream, left, top, width=width, height=height)
            
    prs.save('Matplotlib_Slides_128.pptx')




In [ ]:
bluetopo = "data/raster/BlueTopo.tiff"
flat = "data/raster/Flat_LA.tif"
rough = "data/raster/Rough_FL.tif"
rough_slopey = "data/raster/Rough_Slopey.tif"
slopey = "data/raster/Slopey_MA.tif"

filename = [bluetopo, flat, rough, rough_slopey, slopey]
# methods = ['amp_v1',
#            'psd_v1',
#            'amp_v2',
#             'psd_n',
#             'psd_lf',
#             'psd_df',
#             'spectrum',
#             'spatial_std',
#             'spatial_diff',
#             'spatial_gaussian']
# methods = ['amp_v1',
#            'psd_v1',
#            'amp_v2',
#             'psd_n',
#             'psd_lf',
#             'psd_df']

# filename = [bluetopo]
for filename in filename:
        process_file(filename)